# Assignment 2 — Mini Knowledge Base with Chroma
This notebook builds a local Chroma collection, stores documents with metadata, runs semantic search, visualizes the top retrieved documents, and includes an optional LangChain-based RAG example.


In [ ]:
%pip install -q chromadb sentence-transformers langchain langchain-chroma matplotlib pandas

In [ ]:
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_chroma import Chroma
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer


## 1. Prepare a local Chroma database
The database is stored locally in a folder so the collection persists across runs.


In [ ]:
db_path = "./chroma_assignment2_db"
collection_name = "ai_water_docs"

# Optional reset for a clean rerun
if os.path.exists(db_path):
    shutil.rmtree(db_path)

client = chromadb.PersistentClient(path=db_path)
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)

print(f"Collection created: {collection_name}")


## 2. Add at least 10 documents with metadata
Each document includes text plus metadata fields such as title, source, and type.


In [ ]:
documents = [
    "AI models analyze satellite imagery to detect oil spills and monitor water contamination in oceans.",
    "Machine learning systems classify water quality using sensor readings such as pH, turbidity, and dissolved oxygen.",
    "Computer vision can identify floating plastic waste in rivers from drone footage.",
    "Deep learning helps predict harmful algal blooms using temperature and nutrient data.",
    "Natural language processing extracts environmental findings from scientific water research papers.",
    "AI forecasting models help city planners predict flood risk from rainfall and land use patterns.",
    "Smart monitoring systems use anomaly detection to identify leaks in water distribution pipelines.",
    "Remote sensing with AI helps track changes in wetlands and freshwater ecosystems over time.",
    "Machine learning supports wastewater treatment optimization by adjusting chemical dosing automatically.",
    "AI chat assistants help environmental agencies search knowledge bases about pollution incidents.",
    "Predictive models estimate groundwater contamination spread after industrial chemical discharge.",
    "Semantic search over environmental documents helps retrieve relevant answers about water pollution quickly."
]

metadatas = [
    {"title": "Oil Spill Detection", "source": "scientific", "topic": "pollution"},
    {"title": "Water Quality Sensors", "source": "scientific", "topic": "water_quality"},
    {"title": "Plastic Waste Monitoring", "source": "field_report", "topic": "pollution"},
    {"title": "Algal Bloom Prediction", "source": "scientific", "topic": "forecasting"},
    {"title": "Research Paper Mining", "source": "scientific", "topic": "nlp"},
    {"title": "Flood Risk Forecasting", "source": "government", "topic": "flooding"},
    {"title": "Pipeline Leak Detection", "source": "industry", "topic": "infrastructure"},
    {"title": "Wetland Change Tracking", "source": "scientific", "topic": "ecosystems"},
    {"title": "Wastewater Optimization", "source": "industry", "topic": "treatment"},
    {"title": "Pollution Incident Assistant", "source": "government", "topic": "qa"},
    {"title": "Groundwater Contamination", "source": "scientific", "topic": "pollution"},
    {"title": "Environmental Semantic Search", "source": "technical_blog", "topic": "retrieval"}
]

ids = [f"doc_{i+1}" for i in range(len(documents))]

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Documents in collection:", collection.count())


## 3. Query the collection with 3 different questions
The next cells retrieve the top 3 matching documents for each question.


In [ ]:
queries = [
    "How does AI detect pollution in water bodies?",
    "How can machine learning improve flood prediction?",
    "How is AI used in water infrastructure monitoring?"
]

all_results = {}
for q in queries:
    result = collection.query(query_texts=[q], n_results=3)
    all_results[q] = result
    print(f"\nQuery: {q}")
    for rank, (doc_id, doc, meta, dist) in enumerate(zip(
        result["ids"][0],
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ), start=1):
        print(f"Rank {rank} | ID: {doc_id} | Distance: {dist:.4f}")
        print(f"Title: {meta['title']} | Source: {meta['source']}")
        print(doc)
        print("-" * 80)


## 4. Visualize the retrieved top 3 documents
This produces a simple bar chart for one query and also prints a table for all results. Save the chart or take a screenshot for your deliverable.


In [ ]:
query_to_plot = queries[0]
plot_result = all_results[query_to_plot]

titles = [m["title"] for m in plot_result["metadatas"][0]]
distances = plot_result["distances"][0]

plt.figure(figsize=(10, 5))
plt.barh(titles, distances, color="steelblue")
plt.gca().invert_yaxis()
plt.xlabel("Distance (lower is closer)")
plt.title(f"Top 3 Retrieved Documents for: {query_to_plot}")
plt.tight_layout()
plt.show()


In [ ]:
rows = []
for query, result in all_results.items():
    for rank, (doc_id, meta, doc, dist) in enumerate(zip(
        result["ids"][0],
        result["metadatas"][0],
        result["documents"][0],
        result["distances"][0]
    ), start=1):
        rows.append({
            "query": query,
            "rank": rank,
            "doc_id": doc_id,
            "title": meta["title"],
            "source": meta["source"],
            "distance": dist,
            "document": doc
        })

results_df = pd.DataFrame(rows)
results_df


## 5. Metadata filtering
You can restrict retrieval to only documents that match metadata conditions, such as only `scientific` sources.


In [ ]:
filtered_result = collection.query(
    query_texts=["How does AI detect pollution?"],
    n_results=3,
    where={"source": "scientific"}
)

for rank, (doc_id, doc, meta) in enumerate(zip(
    filtered_result["ids"][0],
    filtered_result["documents"][0],
    filtered_result["metadatas"][0]
), start=1):
    print(f"Rank {rank} | {doc_id} | {meta['title']} | {meta['source']}")
    print(doc)
    print()


## 6. Duplicate text check
Chroma does not automatically deduplicate purely by text content. IDs are what uniquely identify records, so adding the same text twice with different IDs stores both entries.


In [ ]:
duplicate_text = "AI models analyze satellite imagery to detect oil spills and monitor water contamination in oceans."
before_count = collection.count()

collection.add(
    ids=["doc_duplicate"],
    documents=[duplicate_text],
    metadatas=[{"title": "Oil Spill Detection Duplicate", "source": "scientific", "topic": "pollution"}]
)

after_count = collection.count()
print("Count before:", before_count)
print("Count after adding duplicate text with new ID:", after_count)
print("Observation: the count increases, so duplicate text is stored if the ID is new.")


## 7. Optional: LangChain RAG-style retrieval
This uses LangChain with Chroma as the vector store and returns relevant chunks for a question.


In [ ]:
lc_docs = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(documents, metadatas)
]

vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2"),
    collection_name="ai_water_docs_langchain",
    persist_directory="./chroma_langchain_db"
)


In [ ]:
# Alternative LangChain-compatible embeddings wrapper if the previous cell needs adjustment in your environment:
from langchain_community.embeddings import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=hf_embeddings,
    collection_name="ai_water_docs_langchain",
    persist_directory="./chroma_langchain_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
rag_docs = retriever.get_relevant_documents("How does AI help monitor water pollution?")

for i, d in enumerate(rag_docs, start=1):
    print(f"Result {i}: {d.metadata['title']} | {d.metadata['source']}")
    print(d.page_content)
    print()


## Reflection answers
- **Does Chroma deduplicate if the same text is added twice?** Not by text alone. If you add the same text with a new ID, Chroma stores another record.
- **How can you filter by metadata?** Use the `where` argument, for example `where={"source": "scientific"}` to search only scientific documents.
